# Lab 6, Part 3 — Running GPT-type models locally with Ollama (Colab version)

On your own Linux or Mac machine you can skip the first two cells. Install Ollama from https://ollama.com/download/linux, then run `ollama serve` in a terminal.

On Colab a **GPU runtime** is recommended (Runtime → Change runtime type → T4 GPU). A CPU runtime works with 1B-sized models, but it is slow.

In [ ]:
# 1. Install Ollama (Colab / Linux)
!sudo apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 2. Start the server in the background and wait until it responds
import subprocess, time, urllib.request
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2); print("Ollama is up"); break
    except Exception:
        time.sleep(1)

In [ ]:
# 3. Pull two small models to compare (about 1 GB each)
!ollama pull llama3.2:1b
!ollama pull qwen2.5:1.5b
!ollama list

In [ ]:
# 4. A single prompt from the command line
!ollama run llama3.2:1b "Explain what a Transformer is in two sentences." 

## Batch comparison

`ollama_compare.py` (in this folder; on Colab, upload it first) sends the same 9 probe prompts to each local model at temperature 0, with a fixed seed. The prompts cover facts, explanation, reasoning, arithmetic, C++ code, translation, summarisation, a question about a **non-existent paper** (to test hallucination), and strict instruction following.

The script writes `ollama_responses.md`. It contains every answer with its latency and tokens per second, plus empty slots in which to paste the ChatGPT and Claude answers to the same prompts, and a scoring table.

In [ ]:
!python3 ollama_compare.py --model llama3.2:1b --model qwen2.5:1.5b
from IPython.display import Markdown, display
display(Markdown(open("ollama_responses.md").read()))

In [ ]:
# 5. Using Ollama from Python directly (the REST API)
import json, urllib.request
def ollama_chat(model, prompt):
    req = urllib.request.Request("http://localhost:11434/api/chat",
        data=json.dumps({"model": model, "stream": False,
                         "messages": [{"role": "user", "content": prompt}]}).encode(),
        headers={"Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req).read())["message"]["content"]

print(ollama_chat("qwen2.5:1.5b", "Give me one tip for debugging a segmentation fault in C++."))